# NOA

This notebook provides wrapper functions for calling the NOA (Naive Online ALignment) algorithm. Running this algorithm requires installing some other software, which is described below. This notebook implements the `offline_processing()` and `online_processing()` functions, which will be imported and run in `02_RunExperiment.ipynb`.

## Offline Processing

In the offline processing stage, three things are computed and stored in the `cache/` folder:
- Chroma STFT features for the orchestra recording
- Chroma STFT features for the full mix recording

In [ ]:
import os
import numpy as np
import librosa as lb

import system_utils
from noa import alignNOA

In [ ]:
def offline_processing(scenario_dir, cache_dir, hop_length):
    """
    """
    system_utils.verify_scenario_dir(scenario_dir)
    if os.path.exists(cache_dir):
        pass
    else:
        # setup
        os.makedirs(cache_dir)
        
        pref_file = f'{scenario_dir}/pref.wav'
        y_pref, sr = lb.core.load(pref_file)
        F_pref = lb.feature.chroma_stft(y=y_pref, sr=sr, hop_length=hop_length, center=False)
        
        np.save(f'{cache_dir}/pref_stft.npy', F_pref)
        return

In [ ]:
def verify_cache_dir(indir):
    """
    """
    assert os.path.exists(f'{indir}/p_stft.npy'), f'p_stft.npy missing from {indir}'

## Online Processing

In the online processing stage, the following steps are done:
- compute alignment between P_query and P_ref and thus O

### Wrapper Implementation

In [ ]:
def online_processing(scenario_dir, out_dir, cache_dir, hop_length):
    # verify & setup
    system_utils.verify_scenario_dir(scenario_dir)
    verify_cache_dir(cache_dir)
    assert not os.path.exists(out_dir), f'Output directory {out_dir} already exists.'
    os.makedirs(out_dir)
    
    # compute features
    p_file = f'{scenario_dir}/p.wav'
    y, sr = lb.core.load(p_file)
    hop_sec = hop_length / sr
    F_p = lb.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length, center=False)
    
    # compute alignment
    F_pref = np.load(f'{cache_dir}/pref_stft.npy')
    wp = alignNOA(F_p, F_pref)
    
    # TODO: add shifting
    
    np.save(f'{out_dir}/hyp.npy', wp*hop_sec)
    return